# 🏥 HealthBot: AI-Powered Patient Education System
### MediTech Solutions — LangGraph Workflow

This notebook implements a LangGraph-based HealthBot that:
1. Asks the patient for a health topic
2. Searches Tavily for reputable medical information
3. Summarizes findings in patient-friendly language
4. Presents a comprehension quiz
5. Grades the patient's response with cited feedback
6. Allows looping to a new topic or exiting

## Cell 1: Load Environment Variables

In [ ]:
from dotenv import load_dotenv
import os

# Load in the Cohere key and Tavily key.
load_dotenv('config.env')

assert os.getenv('COHERE_API_KEY') is not None, "COHERE_API_KEY not found in config.env"
assert os.getenv('TAVILY_API_KEY') is not None, "TAVILY_API_KEY not found in config.env"

print('✅ API keys loaded successfully.')

## Cell 2: Imports

In [ ]:
from typing import TypedDict, Optional, Annotated
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langchain_cohere import ChatCohere
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage

print('✅ Imports successful.')

## Cell 3: Define HealthBot State

The `HealthBotState` TypedDict defines all shared state fields across the LangGraph nodes.
Each node reads from and writes to this state object.

In [ ]:
class HealthBotState(TypedDict):
    """
    State object shared across all LangGraph nodes.
    Each field is updated by the appropriate node and
    read by subsequent nodes.
    """
    # Conversation messages (tool calls, AI responses, etc.)
    messages: Annotated[list, add_messages]

    # The health topic the patient wants to learn about
    topic: Optional[str]

    # Raw Tavily search results
    search_results: Optional[str]

    # Patient-friendly summarization (3-4 paragraphs)
    summary: Optional[str]

    # Generated quiz question
    quiz_question: Optional[str]

    # Patient's answer to the quiz
    patient_answer: Optional[str]

    # Grade + justification from the model
    grade_and_feedback: Optional[str]

    # Flow control: 'new_topic' or 'exit'
    next_action: Optional[str]

print('✅ HealthBotState defined.')

## Cell 4: Initialize Model and Tavily Tool

In [ ]:
# Initialize the Cohere chat model (free trial tier compatible)
llm = ChatCohere(
    model="command-r-08-2024",
    temperature=0.3,
    cohere_api_key=os.getenv("COHERE_API_KEY")
)

# Initialize Tavily search tool (focused on reputable medical sources)
tavily_tool = TavilySearchResults(
    max_results=5,
    search_depth="advanced",
    include_domains=[
        "mayoclinic.org",
        "webmd.com",
        "medlineplus.gov",
        "healthline.com",
        "nih.gov",
        "cdc.gov",
        "who.int",
        "medicalnewstoday.com",
        "clevelandclinic.org",
        "hopkinsmedicine.org"
    ],
    tavily_api_key=os.getenv("TAVILY_API_KEY")
)

print('✅ LLM (Cohere command-r) and Tavily tool initialized.')
print('   Medical sources: Mayo Clinic, WebMD, NIH, CDC, WHO, and more.')

## Cell 5: Define LangGraph Nodes

Each node has a **single responsibility**. Nodes read from `state` and return updated state fields.

In [ ]:
# ----------------------------------------------------------------
# Node 1: Collect Topic — ask patient what they want to learn
# ----------------------------------------------------------------
def node_collect_topic(state: HealthBotState) -> HealthBotState:
    print("\n" + "="*60)
    print("🏥  Welcome to HealthBot — Your Personal Health Educator")
    print("="*60)
    print("\nI'm here to help you understand medical topics in plain,")
    print("easy-to-understand language. Let's get started!\n")

    topic = input("📋 What health topic or medical condition would you like to learn about today? ")

    print(f"\n🔍 Great! Looking up information on: '{topic}' ...")

    return {
        **state,
        "topic": topic,
        "messages": [HumanMessage(content=f"I want to learn about: {topic}")]
    }


# ----------------------------------------------------------------
# Node 2: Search Tavily — retrieve medical information via tool call
# ----------------------------------------------------------------
def node_search_tavily(state: HealthBotState) -> HealthBotState:
    topic = state["topic"]

    # Bind Tavily tool to LLM so OpenAI can call it
    llm_with_tools = llm.bind_tools([tavily_tool])

    search_prompt = [
        SystemMessage(content=(
            "You are a medical research assistant. Use the tavily_search_results_json "
            "tool to find comprehensive, accurate medical information from reputable health "
            "sources. Find information about what the condition is, its causes, symptoms, "
            "diagnosis, treatment options, and prevention strategies."
        )),
        HumanMessage(content=(
            f"Search for detailed medical information about: {topic}. "
            "Cover causes, symptoms, treatments, and prevention."
        ))
    ]

    # LLM decides to call Tavily
    response = llm_with_tools.invoke(search_prompt)

    # Execute tool calls and collect results
    search_results_text = ""
    tool_call_id = "search_001"

    if response.tool_calls:
        for tool_call in response.tool_calls:
            tool_call_id = tool_call["id"]
            if tool_call["name"] == "tavily_search_results_json":
                raw_results = tavily_tool.invoke(tool_call["args"])
                for i, result in enumerate(raw_results, 1):
                    search_results_text += f"\n[Source {i}]: {result.get('url', 'Unknown')}\n"
                    search_results_text += f"{result.get('content', '')}\n"
                    search_results_text += "-" * 40 + "\n"

    print("✅ Medical information retrieved from reputable sources.")

    return {
        **state,
        "search_results": search_results_text,
        "messages": state["messages"] + [
            response,
            ToolMessage(content=search_results_text, tool_call_id=tool_call_id)
        ]
    }


# ----------------------------------------------------------------
# Node 3: Summarize — produce 3-4 paragraph patient-friendly summary
# ----------------------------------------------------------------
def node_summarize(state: HealthBotState) -> HealthBotState:
    topic = state["topic"]
    search_results = state["search_results"]

    summary_prompt = [
        SystemMessage(content=(
            "You are a compassionate patient educator at a hospital. "
            "Rewrite the provided search results into a clear, warm, patient-friendly summary. "
            "Rules:\n"
            "1. Use ONLY the information provided — do NOT add outside knowledge.\n"
            "2. Write in simple English; explain any medical terms.\n"
            "3. Write exactly 3 to 4 paragraphs.\n"
            "4. Be empathetic, clear, and encouraging.\n"
            "5. Cover: (a) What it is, (b) Causes/Risk Factors, "
               "(c) Symptoms/Diagnosis, (d) Treatment/Prevention.\n"
            "6. End with a reminder to consult a healthcare professional."
        )),
        HumanMessage(content=(
            f"Search results about '{topic}':\n\n"
            f"{search_results}\n\n"
            "Summarize this into a patient-friendly explanation of 3-4 paragraphs "
            "using ONLY the information above."
        ))
    ]

    response = llm.invoke(summary_prompt)
    summary = response.content

    print("✅ Summary generated.")

    return {
        **state,
        "summary": summary,
        "messages": state["messages"] + [response]
    }


# ----------------------------------------------------------------
# Node 4+5: Present Summary — display to patient, wait for readiness
# ----------------------------------------------------------------
def node_present_summary(state: HealthBotState) -> HealthBotState:
    topic = state["topic"]
    summary = state["summary"]

    print("\n" + "="*60)
    print(f"📖  Health Information: {topic.title()}")
    print("="*60)
    print(f"\n{summary}\n")
    print("="*60)
    print("⚠️  Note: This information is for educational purposes only.")
    print("    Always consult your healthcare provider for medical advice.")
    print("="*60)

    # Node 5: Wait for patient to signal readiness
    input("\n✅ Press ENTER when you've finished reading and are ready for a comprehension check...")

    print("\n📝 Great! Generating your quiz question...")

    return state


# ----------------------------------------------------------------
# Node 6: Generate Quiz — one question based only on the summary
# ----------------------------------------------------------------
def node_generate_quiz(state: HealthBotState) -> HealthBotState:
    summary = state["summary"]
    topic = state["topic"]

    quiz_prompt = [
        SystemMessage(content=(
            "You are a medical educator creating a patient comprehension quiz. "
            "Rules:\n"
            "1. Base the question ONLY on information in the provided summary.\n"
            "2. The question must be directly answerable from the summary alone.\n"
            "3. Ask about an important concept (not a trivial detail).\n"
            "4. Write an open-ended question (not multiple choice).\n"
            "5. Output ONLY the question — no preamble, no answer, no explanation."
        )),
        HumanMessage(content=(
            f"Based on this patient education summary about '{topic}':\n\n"
            f"{summary}\n\n"
            "Write ONE comprehension quiz question."
        ))
    ]

    response = llm.invoke(quiz_prompt)
    quiz_question = response.content.strip()

    return {
        **state,
        "quiz_question": quiz_question,
        "messages": state["messages"] + [response]
    }


# ----------------------------------------------------------------
# Node 7+8: Present Quiz — display question, collect patient answer
# ----------------------------------------------------------------
def node_present_quiz(state: HealthBotState) -> HealthBotState:
    quiz_question = state["quiz_question"]

    print("\n" + "="*60)
    print("🧠  Comprehension Check")
    print("="*60)
    print(f"\n❓ {quiz_question}\n")

    patient_answer = input("✏️  Your answer: ")

    print("\n⏳ Evaluating your response...")

    return {
        **state,
        "patient_answer": patient_answer,
        "messages": state["messages"] + [HumanMessage(content=patient_answer)]
    }


# ----------------------------------------------------------------
# Node 9: Grade Answer — letter grade + feedback with summary citations
# ----------------------------------------------------------------
def node_grade_answer(state: HealthBotState) -> HealthBotState:
    summary = state["summary"]
    quiz_question = state["quiz_question"]
    patient_answer = state["patient_answer"]

    grading_prompt = [
        SystemMessage(content=(
            "You are a compassionate medical educator grading a patient's quiz answer. "
            "Rules:\n"
            "1. Grade using ONLY the provided health information summary as your answer key.\n"
            "2. Assign a letter grade: A (excellent), B (good), C (partial), D (minimal), F (incorrect).\n"
            "3. Provide a warm, encouraging explanation.\n"
            "4. Include specific citations from the summary (quote relevant lines) to justify the grade.\n"
            "5. If partially correct, acknowledge what was right before explaining gaps.\n"
            "6. Format your response exactly as:\n"
            "   GRADE: [letter]\n"
            "   FEEDBACK: [your explanation with citations]\n"
            "   CORRECT ANSWER: [brief correct answer from the summary]"
        )),
        HumanMessage(content=(
            f"Health Information Summary:\n{summary}\n\n"
            f"Quiz Question: {quiz_question}\n\n"
            f"Patient's Answer: {patient_answer}\n\n"
            "Grade this answer and provide feedback with citations from the summary."
        ))
    ]

    response = llm.invoke(grading_prompt)
    grade_and_feedback = response.content

    return {
        **state,
        "grade_and_feedback": grade_and_feedback,
        "messages": state["messages"] + [response]
    }


# ----------------------------------------------------------------
# Node 10+11: Present Grade — show results, ask continue or exit
# ----------------------------------------------------------------
def node_present_grade(state: HealthBotState) -> HealthBotState:
    grade_and_feedback = state["grade_and_feedback"]

    print("\n" + "="*60)
    print("📊  Your Results")
    print("="*60)
    print(f"\n{grade_and_feedback}\n")
    print("="*60)
    print("\n💙 Great effort! Learning about your health is the first step toward better wellbeing.")

    print("\n" + "-"*60)
    print("What would you like to do next?")
    print("  1. Learn about a new health topic")
    print("  2. Exit HealthBot")
    print("-"*60)

    while True:
        choice = input("\n👉 Enter 1 or 2: ").strip()
        if choice == "1":
            next_action = "new_topic"
            break
        elif choice == "2":
            next_action = "exit"
            break
        else:
            print("   Please enter 1 or 2.")

    return {
        **state,
        "next_action": next_action
    }


# ----------------------------------------------------------------
# Node 12a: Reset State — clear all data before a new topic
# ----------------------------------------------------------------
def node_reset_state(state: HealthBotState) -> HealthBotState:
    print("\n" + "="*60)
    print("🔄  Starting a new session...")
    print("="*60)
    print("✅ Previous session data cleared for your privacy.\n")

    # Return a completely fresh state — no previous data carries over
    return {
        "messages": [],
        "topic": None,
        "search_results": None,
        "summary": None,
        "quiz_question": None,
        "patient_answer": None,
        "grade_and_feedback": None,
        "next_action": None
    }


# ----------------------------------------------------------------
# Node 12b: Exit — end the session gracefully
# ----------------------------------------------------------------
def node_exit(state: HealthBotState) -> HealthBotState:
    print("\n" + "="*60)
    print("👋  Thank you for using HealthBot!")
    print("="*60)
    print("\nStay informed, stay healthy. Always consult your healthcare")
    print("provider for personalized medical advice.")
    print("\nGoodbye! 💙\n")
    return state


print('✅ All 10 nodes defined successfully.')

## Cell 6: Define Routing Logic (Conditional Edge)

In [ ]:
def route_after_grade(state: HealthBotState) -> str:
    """
    Conditional edge router: After the grade is presented,
    routes to 'reset_state' (new topic) or 'exit' based on patient choice.
    """
    if state.get("next_action") == "new_topic":
        return "reset_state"
    else:
        return "exit"

print('✅ Routing function defined.')

## Cell 7: Build the LangGraph Workflow

```
collect_topic
     ↓
search_tavily
     ↓
  summarize
     ↓
present_summary
     ↓
generate_quiz
     ↓
present_quiz
     ↓
grade_answer
     ↓
present_grade
    / \
   ↙   ↘
reset  exit
  ↓      ↓
collect  END
```

In [ ]:
def build_healthbot_graph():
    graph = StateGraph(HealthBotState)

    # --- Add all nodes ---
    graph.add_node("collect_topic",   node_collect_topic)
    graph.add_node("search_tavily",   node_search_tavily)
    graph.add_node("summarize",       node_summarize)
    graph.add_node("present_summary", node_present_summary)
    graph.add_node("generate_quiz",   node_generate_quiz)
    graph.add_node("present_quiz",    node_present_quiz)
    graph.add_node("grade_answer",    node_grade_answer)
    graph.add_node("present_grade",   node_present_grade)
    graph.add_node("reset_state",     node_reset_state)
    graph.add_node("exit",            node_exit)

    # --- Set entry point ---
    graph.set_entry_point("collect_topic")

    # --- Sequential edges (main workflow) ---
    graph.add_edge("collect_topic",   "search_tavily")
    graph.add_edge("search_tavily",   "summarize")
    graph.add_edge("summarize",       "present_summary")
    graph.add_edge("present_summary", "generate_quiz")
    graph.add_edge("generate_quiz",   "present_quiz")
    graph.add_edge("present_quiz",    "grade_answer")
    graph.add_edge("grade_answer",    "present_grade")

    # --- Conditional edge: new topic OR exit ---
    graph.add_conditional_edges(
        "present_grade",
        route_after_grade,
        {
            "reset_state": "reset_state",
            "exit":        "exit"
        }
    )

    # --- After reset, loop back to start ---
    graph.add_edge("reset_state", "collect_topic")

    # --- Exit leads to END ---
    graph.add_edge("exit", END)

    return graph.compile()


healthbot = build_healthbot_graph()

print('✅ HealthBot LangGraph compiled successfully!')
print('\nWorkflow nodes:')
for node in [
    'collect_topic', 'search_tavily', 'summarize',
    'present_summary', 'generate_quiz', 'present_quiz',
    'grade_answer', 'present_grade', 'reset_state', 'exit'
]:
    print(f'  • {node}')

## Cell 8: Run the HealthBot 🚀

Execute the cell below to start your HealthBot session!

In [ ]:
# Fresh initial state — all fields start as None
initial_state: HealthBotState = {
    "messages": [],
    "topic": None,
    "search_results": None,
    "summary": None,
    "quiz_question": None,
    "patient_answer": None,
    "grade_and_feedback": None,
    "next_action": None
}

print("🏥 " * 20)
print("    Starting HealthBot Session...")
print("🏥 " * 20 + "\n")

# Run the graph — it blocks on input() calls and loops until the patient exits
final_state = healthbot.invoke(initial_state)

print("\n" + "="*60)
print("Session complete. Final state summary:")
print(f"  Last topic: {final_state.get('topic')}")
print(f"  Messages in history: {len(final_state.get('messages', []))}")
print("="*60)